# Contact Examples

This is a jupyter notebook. The "real" notebook can be found [here](https://github.com/polyfem/polyfem.github.io/blob/docs/docs/contact.ipynb).

Some imports for plotting

In [ ]:
import meshplot as mp
import numpy as np
import polyfempy as pf

## Simple

Setup problem, set `contact_problem` to `True`

In [ ]:
settings = pf.Settings(
    contact_problem=True,
    
    tend=1,
    time_steps=10,
    
    pde=pf.PDEs.NonLinearElasticity
)

settings.set_material_params("E", 200000)
settings.set_material_params("nu", 0.3)
settings.set_material_params("rho", 1000)

problem = pf.Problem(is_time_dependent=True)
problem.set_dirichlet_value(3, [0, 0, 0])
problem.set_dirichlet_value(1, [1, 0, 0])

settings.problem = problem

Get sideset and show problem

In [ ]:
solver = pf.Solver()
solver.settings(settings)

solver.load_mesh_from_path("cubes.mesh", vismesh_rel_area=0.001, normalize_mesh=False)
p, t, s = solver.get_boundary_sidesets()


tmp = np.zeros_like(s)
tmp[s==3] = 1
tmp[s==1] = 2
mp.plot(p, t, tmp)

Solve

In [ ]:
solver.solve()

Plot

In [ ]:
pts = solver.get_sampled_points_frames()
tris = solver.get_sampled_connectivity_frames()
displacement = solver.get_sampled_solution_frames()

In [ ]:
def plot(frame, p=None):
    pp = pts[frame]
    tt = tris[frame]
    dd = displacement[frame]
    p1 = mp.plot(pp+dd, tt, np.linalg.norm(dd, axis=1), plot=p)
    if p is None:
        p=p1
    return p

In [ ]:
plot(9)

In [ ]:
p = plot(0)

@mp.interact(frame=(0, len(displacement)-1))
def step(frame=0):
    plot(frame, p)

## Multimaterial with contact

In [ ]:
settings = pf.Settings(
    contact_problem=True,
    dhat=0.01,
    
    tend=0.5,
    time_steps=10,
    
    pde=pf.PDEs.NonLinearElasticity
)

settings.set_body_params(1, E=1e8, nu=0.4, rho=2000)
settings.set_body_params(2, E=1e6, nu=0.4, rho=1000)

problem = pf.Problem(is_time_dependent=True)
problem.set_dirichlet_value(2, [0, 0, 0])
problem.rhs = [0, 9.81, 0]

settings.problem = problem

Creaset a selection to assign sidesets and body ids

In [ ]:
selection = pf.Selection()

#Assign body 1 to the sphere and body 2 to the mat
selection.select_body_with_sphere(1, center=[0, 1, 0], radius=0.6)
selection.select_body_with_axis_plane(2, axis=-2, position=0.01)

#Assign sideset 2 to the four sides of the mat
selection.select_sideset_with_axis_plane(2, axis=-1, position=-0.99)
selection.select_sideset_with_axis_plane(2, axis=1, position=0.99)
selection.select_sideset_with_axis_plane(2, axis=-3, position=-0.99)
selection.select_sideset_with_axis_plane(2, axis=3, position=0.99)

settings.selection = selection

In [ ]:
solver = pf.Solver()
solver.settings(settings)

solver.load_mesh_from_path("sphere-mat.msh", vismesh_rel_area=0.001, normalize_mesh=False)

# Only one sideset exists, 2
p, t, s = solver.get_boundary_sidesets()
mp.plot(p, t, s)

In [ ]:
solver.solve()

In [ ]:
pts = solver.get_sampled_points_frames()
tris = solver.get_sampled_connectivity_frames()
displacement = solver.get_sampled_solution_frames()

In [ ]:
plot(9)

In [ ]:
p = plot(0)

@mp.interact(frame=(0, len(displacement)-1))
def step(frame=0):
    plot(frame, p)